# Step 1 — Data Preprocessing

Extracts frames from each video in the [Real Life Violence Situations Dataset](https://www.kaggle.com/datasets/mohamedmustafa/real-life-violence-situations-dataset) and saves the result as a NumPy archive (`dataset.npz`) for fast reloading during training.

**Expected dataset structure on Kaggle:**
```
/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset/
    NonViolence/
        NV_1.mp4
        NV_2.mp4  ...
    Violence/
        V_1.mp4
        V_2.mp4   ...
```

**Output shape:** `(N, 60, 64, 64, 3)` — N videos × 60 frames × 64×64 pixels × RGB

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/real-life-violence-situations-dataset/Real Life Violence Dataset"
IMG_H      = 64          # frame height (pixels)
IMG_W      = 64          # frame width  (pixels)
SEQ_LEN    = 60          # number of frames sampled per video
CLASSES    = ["NonViolence", "Violence"]   # index 0 = NonViolence, 1 = Violence
OUTPUT_NPZ = "dataset.npz"

In [ ]:
def extract_frames(video_path, seq_len=SEQ_LEN, img_h=IMG_H, img_w=IMG_W):
    """
    Uniformly sample `seq_len` frames from a video file.

    Sampling interval is computed from the video duration so that
    frames are spread evenly regardless of video length or FPS.
    If the video yields seq_len-1 frames (off-by-one edge case),
    the last frame is duplicated to reach the required length.

    Args:
        video_path : path to video file
        seq_len    : number of frames to extract
        img_h/w    : resize target dimensions

    Returns:
        list of (img_h, img_w, 3) uint8 arrays, length == seq_len
        or empty list if the video cannot be read / is too short
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return []

    fps         = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    duration    = frame_count / fps if fps > 0 else 0

    if duration < 1.0:        # skip videos shorter than 1 second
        cap.release()
        return []

    interval_ms = (duration * 1000) / seq_len   # ms between samples
    frames      = []
    timestamp   = 0.0

    while timestamp < duration * 1000 and len(frames) < seq_len:
        cap.set(cv2.CAP_PROP_POS_MSEC, timestamp)
        success, frame = cap.read()
        if not success:
            break
        frame = cv2.resize(frame, (img_w, img_h))
        frames.append(frame)
        timestamp += interval_ms

    cap.release()

    # Edge case: off-by-one — duplicate last frame
    if len(frames) == seq_len - 1:
        frames.append(frames[-1])

    return frames

In [ ]:
def build_dataset(data_dir, classes=CLASSES, seq_len=SEQ_LEN):
    """
    Walk the dataset directory and build (X, Y) arrays.

    Each class subdirectory name must match an entry in `classes`.
    Labels are one-hot encoded: NonViolence → [1,0], Violence → [0,1].
    Videos that yield fewer than seq_len frames are skipped.

    Returns:
        X : np.ndarray  shape (N, seq_len, IMG_H, IMG_W, 3)  float32
        Y : np.ndarray  shape (N, len(classes))              float32
    """
    X, Y = [], []
    total = 0

    for cls in os.listdir(data_dir):
        cls_path = os.path.join(data_dir, cls)
        if not os.path.isdir(cls_path) or cls not in classes:
            continue
        print(f"Processing class: {cls}")

        for video_file in os.listdir(cls_path):
            video_path = os.path.join(cls_path, video_file)
            frames     = extract_frames(video_path, seq_len)

            if len(frames) != seq_len:
                continue          # skip corrupted / too-short videos

            X.append(frames)
            label    = [0] * len(classes)
            label[classes.index(cls)] = 1
            Y.append(label)
            total += 1

            if total % 50 == 0:
                print(f"  {total} videos processed…")

    X = np.asarray(X, dtype=np.float32) / 255.0   # normalise to [0, 1]
    Y = np.asarray(Y, dtype=np.float32)
    print(f"\nDone — {total} videos loaded.")
    return X, Y

In [ ]:
# ── Run preprocessing ─────────────────────────────────────────────────────────
X, Y = build_dataset(DATA_DIR)
print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

In [ ]:
# ── Save to disk ──────────────────────────────────────────────────────────────
# Frame extraction is expensive (~hours on CPU). Save once, reload in seconds.
np.savez_compressed(OUTPUT_NPZ, X=X, Y=Y)
print(f"Saved to {OUTPUT_NPZ}")